<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/human_validation_sampling_hellaswag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import json
import random
import pandas as pd
from google.colab import files


# -------------------------
# Settings
# -------------------------

RANDOM_SEED = 42
SAMPLE_SIZE = 10
BENCHMARK_NAME = "HellaSwag"

FILE_PATHS = {
    "0.1": "/content/hellaswag_rate_0.1.jsonl",
    "0.4": "/content/hellaswag_rate_0.4.jsonl",
    "0.7": "/content/hellaswag_rate_0.7.jsonl"
}


# -------------------------
# Read JSONL files
# -------------------------

def read_jsonl(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                record["_source_row"] = line_number
                records.append(record)

            except json.JSONDecodeError as error:
                print(
                    f"Invalid JSON on line {line_number} "
                    f"in {file_path}: {error}"
                )

    return records


# -------------------------
# Format HellaSwag choices
# -------------------------

def format_endings(endings):
    labels = ["A", "B", "C", "D"]

    return "\n".join(
        f"{labels[index]}. {ending}"
        for index, ending in enumerate(endings)
    )


# -------------------------
# Convert HellaSwag answer
# -------------------------

def format_gold_answer(label):
    label_map = {
        0: "A",
        1: "B",
        2: "C",
        3: "D",
        "0": "A",
        "1": "B",
        "2": "C",
        "3": "D"
    }

    return label_map.get(label, label)


# -------------------------
# Load all typo-rate datasets
# -------------------------

datasets = {}

for typo_rate, file_path in FILE_PATHS.items():
    datasets[typo_rate] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[typo_rate])} records "
        f"for typo rate {typo_rate}"
    )


# -------------------------
# Create lookup tables
# -------------------------

record_maps = {}

for typo_rate, records in datasets.items():
    record_maps[typo_rate] = {
        record["original_text_backup"]: record
        for record in records
        if record.get("original_text_backup")
    }


# -------------------------
# Find prompts shared by all three rates
# -------------------------

common_original_prompts = set(record_maps["0.1"].keys())

for typo_rate in ["0.4", "0.7"]:
    common_original_prompts = common_original_prompts.intersection(
        record_maps[typo_rate].keys()
    )

common_original_prompts = sorted(common_original_prompts)

print(f"Common original prompts: {len(common_original_prompts)}")


if len(common_original_prompts) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(common_original_prompts)} matched prompts were found. "
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# -------------------------
# Randomly sample original prompts
# -------------------------

random.seed(RANDOM_SEED)

selected_original_prompts = random.sample(
    common_original_prompts,
    SAMPLE_SIZE
)

print(f"Random seed: {RANDOM_SEED}")
print(f"Selected original prompts: {SAMPLE_SIZE}")


# -------------------------
# Build the annotation table
# -------------------------

output_rows = []

for prompt_number, original_prompt in enumerate(
    selected_original_prompts,
    start=1
):
    base_id = f"{BENCHMARK_NAME}_{prompt_number:03d}"

    for typo_rate in ["0.1", "0.4", "0.7"]:
        record = record_maps[typo_rate][original_prompt]

        output_rows.append({
            "Base_ID": base_id,
            "Sample_ID": f"{base_id}_rate_{typo_rate}",
            "Benchmark": BENCHMARK_NAME,
            "Typo_Rate": typo_rate,
            "Source_Row": record.get("_source_row", ""),

            "Original_Prompt": original_prompt,
            "Modified_Prompt": record.get("ctx", ""),

            "Answer_Choices": format_endings(
                record.get("endings", [])
            ),

            "Gold_Answer": format_gold_answer(
                record.get("label", "")
            ),

            "R1_Meaning": "",
            "R2_Meaning": "",
            "Final_Meaning": "",

            "R1_Key_Info": "",
            "R2_Key_Info": "",
            "Final_Key_Info": "",

            "R1_Realism": "",
            "R2_Realism": "",
            "Final_Realism": "",

            "R1_Readability": "",
            "R2_Readability": "",
            "Final_Readability": "",

            "Comments": ""
        })


sample_df = pd.DataFrame(output_rows)

print(f"Total output rows: {len(sample_df)}")

display(sample_df)


# -------------------------
# Save the CSV file
# -------------------------

output_file = "/content/HellaSwag_matched_sample_n10_seed42.csv"

sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved successfully: {output_file}")


# -------------------------
# Download the CSV file
# -------------------------

files.download(output_file)

Invalid JSON on line 6107 in /content/hellaswag_rate_0.1.jsonl: Unterminated string starting at: line 1 column 594 (char 593)
Loaded 6106 records for typo rate 0.1
Invalid JSON on line 6116 in /content/hellaswag_rate_0.4.jsonl: Expecting value: line 1 column 587 (char 586)
Loaded 6115 records for typo rate 0.4
Invalid JSON on line 4888 in /content/hellaswag_rate_0.7.jsonl: Unterminated string starting at: line 1 column 264 (char 263)
Loaded 4887 records for typo rate 0.7
Common original prompts: 4887
Random seed: 42
Selected original prompts: 10
Total output rows: 30


,Base_ID,Sample_ID,Benchmark,Typo_Rate,Source_Row,Original_Prompt,Modified_Prompt,Answer_Choices,Gold_Answer,R1_Meaning,...,R1_Key_Info,R2_Key_Info,Final_Key_Info,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,Comments
0,HellaSwag_001,HellaSwag_001_rate_0.1,HellaSwag,0.1,3059,A man enters a room and piles plaster onto a b...,A man enters a room and piles plaser onto a ba...,A. puts red latex over the strip and is done.\...,C,,...,,,,,,,,,,
1,HellaSwag_001,HellaSwag_001_rate_0.4,HellaSwag,0.4,3059,A man enters a room and piles plaster onto a b...,A man enters a room and pols plaster onot a ba...,A. puts red latex over the strip and is done.\...,C,,...,,,,,,,,,,
2,HellaSwag_001,HellaSwag_001_rate_0.7,HellaSwag,0.7,3059,A man enters a room and piles plaster onto a b...,A mab enters a roj amd piles plastrer oto a bq...,A. puts red latex over the strip and is done.\...,C,,...,,,,,,,,,,
3,HellaSwag_002,HellaSwag_002_rate_0.1,HellaSwag,0.1,4521,A bull runs through a crowd. the people,A bull runs through a croed. the people,A. aim bows and arrow to shoot at the bull.\nB...,B,,...,,,,,,,,,,
4,HellaSwag_002,HellaSwag_002_rate_0.4,HellaSwag,0.4,4521,A bull runs through a crowd. the people,A bull runs throuhg a crod. the peoplr,A. aim bows and arrow to shoot at the bull.\nB...,B,,...,,,,,,,,,,
5,HellaSwag_002,HellaSwag_002_rate_0.7,HellaSwag,0.7,4521,A bull runs through a crowd. the people,A buk runs throguh a crow. the peopel,A. aim bows and arrow to shoot at the bull.\nB...,B,,...,,,,,,,,,,
6,HellaSwag_003,HellaSwag_003_rate_0.1,HellaSwag,0.1,1818,A woman sitting in a room puts on makeup as a ...,A woman sitting ib a room puts on makeup as a ...,A. takes a black and red liquid and rubs it al...,C,,...,,,,,,,,,,
7,HellaSwag_003,HellaSwag_003_rate_0.4,HellaSwag,0.4,1818,A woman sitting in a room puts on makeup as a ...,A woman sitting in a rom pus on makeup as a so...,A. takes a black and red liquid and rubs it al...,C,,...,,,,,,,,,,
8,HellaSwag_003,HellaSwag_003_rate_0.7,HellaSwag,0.7,1818,A woman sitting in a room puts on makeup as a ...,A womna sititn in a rii pus on makeup as a son...,A. takes a black and red liquid and rubs it al...,C,,...,,,,,,,,,,
9,HellaSwag_004,HellaSwag_004_rate_0.1,HellaSwag,0.1,4407,A woman helps a little boy kick a ball during ...,A woman helps a little boy kick a ball during ...,"A. , the little boy hits it with both feet.\nB...",D,,...,,,,,,,,,,


CSV saved successfully: /content/HellaSwag_matched_sample_n10_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>